<center><ins><h1>Suspended Solids</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Volatile and Total suspended solids (VSS/TSS) are analytical parameters representing, loosely, the undissolved organic matter in a water sample. More technically, they are water quality parameters and a factor for growth.</li> 
    <li>Formula for TSS g/L = ((DRIEDFILTER_G - EMPTYFILTER_G) / VOLUME_ML) * 1000. TSS can be further used for sludge indices.</li>
    <li>Formula for VSS g/L = ((BURNEDFILTER_G - EMPTYFILTER_G + BLANKFILTER_G) / VOLUME_ML) * 1000.</li>
    <li>Import of only one big data frame .xlsx.</li>
</ul>

<center>
<img src="../images/growth_device.png" alt="Device" width="225" height="300">
<img src="../images/growth_diagram.png" alt="Diagram" width="533" height="300">
<img src="../images/growth_example-graph.png" alt="Example-Graph" width="579" height="300">
</center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Packages

In [1]:
# Imports
import os, sys
sys.path.append(os.path.abspath("/Users/cedric/Documents/Career/PhD-Student/Experiments/Data-Science"))
from scripts import data_helper, plot_helper
import pandas as pd

# Variables

In [2]:
META_DATA = pd.read_csv("../data/meta_data.csv")

# Show numeric output in decimal format e.g., 2.1514
pd.options.display.float_format = '{:,.4f}'.format

# Functions

In [3]:
def custom_solid_import(file, sample_col, value_cols):
    master_df = pd.read_excel(file)
    master_df.columns = data_helper.formatting_strings(master_df.columns) # All columns uppercase

    raw_df = pd.DataFrame()
    # add data value(s) into clean data frame
    for index, name in enumerate(value_cols):
        raw_df.insert(index, name, master_df[name])

    # divide and add sample information into clean data frame
    for index, name in enumerate(sample_col):
        new_column = master_df[sample_col[index]]
        raw_df.insert(index, name, new_column)
    
    return raw_df

# Raw Import

In [4]:
# Define data path and get all associated files
data_dir = "/Users/cedric/Documents/Career/PhD-Student/Experiments/Instrument-Data/Fine-Scale_Sartorius/CH_Master/CH_Master_TSS+VSS.xlsx"
#files = data_helper.create_filelist(data_dir, skip = " ")

# Import raw data frame
raw_df = custom_solid_import(data_dir, 
                             sample_col = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "BIO_REP", "TEC_REP"],
                             value_cols = ["EMPTYFILTER_G", "VOLUME_ML", "DRIEDFILTER_G", "BURNEDFILTER_G", "NOTE"])
raw_df.SAMPLE_NAME = raw_df.SAMPLE_NAME.astype(str) # Double convert to string for some reason

# Convert data types
raw_df = data_helper.convert_types(raw_df)
# Append start controls to physiology samples    
raw_df = data_helper.copy_start_control(raw_df, META_DATA)
# Sort data
raw_df = raw_df.sort_values(by = ["EXPERIMENT_NAME", "TIME", "SAMPLE_NAME", "BIO_REP", "TEC_REP"], ignore_index = True)
raw_df.head(3)

,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,EMPTYFILTER_G,VOLUME_ML,DRIEDFILTER_G,BURNEDFILTER_G,NOTE
0,CH230710,AT,0.1,0,1,1,0.0903,20,0.1161,0.0905,0
1,CH230802,AT,0.1,0,2,1,0.0930,15,0.1095,0.0929,0
2,CH230907,AT,0.1,0,3,1,0.0922,15,0.1115,0.0921,0


# Custom Cleaning

In [5]:
clean_df = pd.DataFrame(raw_df)

# Clean from stock, reactor and test samples
clean_df = clean_df.drop(clean_df[clean_df["SAMPLE_NAME"].str.contains("Reactor", na=False)].index)
clean_df = clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "Test"].index)

# Get and calculate mean VSS-Blank and drop blank afterwards off clean_df
blank_mean = (clean_df[clean_df["SAMPLE_NAME"] == "VSS-Blank"]["EMPTYFILTER_G"] - clean_df[clean_df["SAMPLE_NAME"] == "VSS-Blank"]["BURNEDFILTER_G"]).mean()
clean_df = clean_df.drop(clean_df[clean_df["NOTE"].str.contains("Blank", na=False) | clean_df["NOTE"].str.contains("Drop", na=False)].index)

# Lower species count from 7 to 5
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "C.vulgaris"].index, inplace = True)
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "P.duplex"].index, inplace = True)

print(f"{raw_df.shape[0] - clean_df.shape[0]} rows and {raw_df.shape[1] - clean_df.shape[1]} cols were cleaned.")

255 rows and 0 cols were cleaned.


# Data Calculation

In [6]:
# Initialize new data frame for calculation
calc_df = clean_df.iloc[:, 0:6]

# Calculate TSS
calc_df["TSS_BIOMASS_G"] = clean_df["DRIEDFILTER_G"] - clean_df["EMPTYFILTER_G"]
calc_df["TSS_g/L"] = (calc_df["TSS_BIOMASS_G"] / clean_df["VOLUME_ML"] * 1000).astype("Float64")

# Prepare VSS Data
calc_df["VSS_BIOMASS_G"] = clean_df["BURNEDFILTER_G"] - clean_df["EMPTYFILTER_G"] + blank_mean # calculated from all VSS-Blanks in clean_df
# Filter for negative VSS values and set to 0
calc_df.loc[calc_df["VSS_BIOMASS_G"] < 0, ["VSS_BIOMASS_G"]] = 0
# Calculate VSS
calc_df["VSS_g/L"] = (calc_df["VSS_BIOMASS_G"] / clean_df["VOLUME_ML"] * 1000).astype("Float64")

# Calculate TSS / VSS percentages
calc_df["TSS_%"] = (100 - ((calc_df["VSS_g/L"] / calc_df["TSS_g/L"]) * 100)).astype("Float64")
calc_df["VSS_%"] = ((calc_df["VSS_g/L"] / calc_df["TSS_g/L"]) * 100).astype("Float64")

calc_df.head(3)

,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,TSS_BIOMASS_G,TSS_g/L,VSS_BIOMASS_G,VSS_g/L,TSS_%,VSS_%
0,CH230710,AT,0.1,0,1,1,0.0258,1.2900,0.0012,0.0615,95.2309,4.7691
1,CH230802,AT,0.1,0,2,1,0.0165,1.1000,0.0009,0.0620,94.3610,5.6390
2,CH230907,AT,0.1,0,3,1,0.0193,1.2867,0.0009,0.0620,95.1791,4.8209


# File Export

In [7]:
# Export to raw data to csv-file
export_df = pd.DataFrame(calc_df)
file_name = "../../OUTPUT/Raw-Data/Suspended-Solids_Raw-Data.csv"
export_df.to_csv(file_name, encoding = "utf-8", index = False)

# Plot Visualization

In [8]:
# Filter time before plotting
plot_df = pd.DataFrame(data_helper.filter(calc_df, time = [12]))

# Iterate through all experiment, visualize and save absolute plots
for exp in META_DATA["EXP_ABBR"].unique():
    sub_df = plot_df[plot_df["EXPERIMENT_NAME"] == exp]
    sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
    
    plot_helper.visualize_barplot(sub_df, sub_meta, # current data subset
                          y_data = ["TSS_g/L", "VSS_g/L"],
                          y_label = "Suspended Solids [g\u0020" + r"${L^{-1}}$]", # Without unit serves as file name
                          y_ticks = [0, 3, 0.5],
                          y_labelpad = 13,)
    
# # Iterate through all experiment, visualize and save relative plots
# for exp in META_DATA["EXP_ABBR"].unique():
#     sub_df = plot_df[plot_df["EXPERIMENT_NAME"] == exp]
#     sub_df.loc[:,"TSS_%"] = 100
#     sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
    
#     plot_helper.visualize_barplot(sub_df, sub_meta, # current data subset
#                           y_data = ["TSS_%", "VSS_%"],
#                           y_label = "Suspended Solids [%]", # Without unit serves as file name
#                           y_ticks = [0, 110, 10],
#                           y_labelpad = 7,)

Abs-Barplot "Species Composition" created and saved.
Abs-Barplot "Salinity Treatment" created and saved.
Abs-Barplot "pH Treatment" created and saved.
Abs-Barplot "Antibiotics Treatment" created and saved.
Abs-Barplot "Light Treatment" created and saved.
Abs-Barplot "Temperature Treatment" created and saved.
Abs-Barplot "Culture Composition" created and saved.
Abs-Barplot "Reflocculation" created and saved.
Abs-Barplot "Bioflocculation" created and saved.


# ARCHIVE